# 📘 Notebook 4: Training GPT from Scratch — Basics to Scale
## *From your first training loop to multi-GPU distributed training*

**Course:** Building LLMs from Scratch  
**Module:** MyLLM — Your Own Language Model  
**Part of:** SAIR ML/DL Roadmap & Bootcamp 🇸🇩

---

## 🌱 Why This Notebook Matters

In Notebook 3 you built the GPT architecture — attention, transformer blocks, the full stack.  
In Notebook 1 you built the data pipeline — tokens, sliding windows, DataLoader.

Now we bring them together and actually **train** the model.

This notebook has two phases:

```
PHASE 1 — Basics (CPU-friendly)
    ├── Load the data pipeline (from Notebook 1)
    ├── Sanity-check: overfit on one batch
    ├── Build trainerV1 — the clean training loop
    └── Watch the model learn to generate text

PHASE 2 — Scale (GPU-optimized)
    ├── Section 1: GPU Performance  (TF32, BF16 mixed precision, torch.compile, Flash Attention)
    ├── Section 2: Algorithmic Tuning  (AdamW, cosine LR, gradient clipping, accumulation)
    └── Section 3: Distributed Training  (DDP, torchrun, multi-GPU)
```

By the end, you will have gone from a single-CPU training loop to a production-grade
distributed trainer that matches the techniques used to train GPT-2 and GPT-3.

---

## 📍 Where You Are in the MyLLM Journey

```
MyLLM Module
    ├── Notebook 1: Data Pipeline          ✅
    ├── Notebook 2: Tokenizer & Dataset    ✅
    ├── Notebook 3: GPT Architecture       ✅
    ├── Notebook 4: Training               ← YOU ARE HERE
    └── Notebook 5: Fine-Tuning & Inference
```

## ⚙️ Setup

In [2]:
# Uncomment if running in Google Colab
# !pip install tiktoken

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import tiktoken
import time
import os

---
## 🌉 The Bridge — What You Already Built

🔗 **Notebook 1 Connection:** You built `GPT2Dataset` — a sliding-window Dataset that reads
tokenized `.bin` files and returns `(input_ids, target_ids)` pairs.

🔗 **Notebook 3 Connection:** You built `GPTModel` with multi-head attention and transformer
blocks. In Phase 2 we upgrade the attention layer to **Flash Attention**.

**The new thing in this notebook:** the training loop — loss, backward pass, optimizer,
evaluation, text generation, and all the optimizations that make it scale.

---
# 📦 PART 0: Data Pipeline (from Notebook 1)

We re-use the `GPT2Dataset` class you built in Notebook 1.
This sliding-window dataset reads a flat `.bin` token stream and returns
`(input_ids, target_ids)` pairs where the target is the input shifted by 1 —
the **next-token prediction** objective.

In [3]:
class GPT2Dataset(Dataset):
    """
    Sliding-window dataset over a flat token stream.
    Reads a .bin file of int32 token IDs.
    Target = input shifted by 1  →  next-token prediction.
    """
    def __init__(self, file_path, max_length, stride):
        self.data       = np.fromfile(file_path, dtype=np.int32)
        self.max_length = max_length
        self.stride     = stride

    def __len__(self):
        return (len(self.data) - self.max_length) // self.stride

    def __getitem__(self, idx):
        start      = idx * self.stride
        input_seq  = torch.tensor(self.data[start     : start + self.max_length],     dtype=torch.long)
        output_seq = torch.tensor(self.data[start + 1 : start + self.max_length + 1], dtype=torch.long)
        return input_seq, output_seq

## Model Configurations

We define two configs:
- **`GPT_CONFIG_SMALL`** — lightweight, CPU-trainable, used in Phase 1
- **`GPT_CONFIG_124M`** — full GPT-2 size, used in Phase 2 with GPU

### Why reduce for CPU?

| Parameter | GPT-2 124M | CPU-Friendly | Reason |
|-----------|-----------|--------------|--------|
| `context_length` | 1024 | 256 | Cuts memory quadratically (attention is O(T²)) |
| `emb_dim` | 768 | 512 | Fewer parameters → faster matrix multiplications |
| `n_layers` | 12 | 6 | Halves depth → 2x faster forward+backward pass |
| `n_heads` | 12 | 8 | Lower per-layer compute |
| `vocab_size` | 50257 | 50257 | Must match tokenizer — never change this |

In [4]:
# CPU-friendly config — reduced depth and width for fast iteration
GPT_CONFIG_SMALL = {
    "vocab_size":     50257,
    "context_length": 256,
    "emb_dim":        512,
    "n_heads":        8,
    "n_layers":       6,
    "drop_rate":      0.1,
    "qkv_bias":       False,
}

# Full GPT-2 124M config — use with a CUDA GPU
GPT_CONFIG_124M = {
    "vocab_size":     50257,
    "context_length": 1024,
    "emb_dim":        768,
    "n_head":         12,
    "n_layer":        12,
    "dropout":        0.1,
    "qkv_bias":       False,
}

## Loading the Datasets & Creating DataLoaders

In [5]:
train_path = "data/train_ids.bin"
val_path   = "data/val_ids.bin"
test_path  = "data/test_ids.bin"

CONTEXT  = GPT_CONFIG_SMALL["context_length"]   # 256 tokens
BATCH    = 2
WORKERS  = 0   # 0 = single-threaded (safe on CPU)

train_dataset = GPT2Dataset(train_path, max_length=CONTEXT, stride=CONTEXT)
val_dataset   = GPT2Dataset(val_path,   max_length=CONTEXT, stride=CONTEXT)
test_dataset  = GPT2Dataset(test_path,  max_length=CONTEXT, stride=CONTEXT)

train_loader  = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=WORKERS)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=WORKERS)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=WORKERS)

# Sanity check
for x, y in train_loader:
    print(f"Input shape:  {x.shape}   (batch_size, context_length)")
    print(f"Target shape: {y.shape}   (same — shifted by 1)")
    break

FileNotFoundError: [Errno 2] No such file or directory: 'data/train_ids.bin'

<details>
<summary>🧾 Whiteboard Cue — draw the full data flow</summary>

```
.bin file  (flat token stream: [tok0, tok1, tok2, ...])
    ↓  GPT2Dataset (sliding window, stride=context_length)
(input_ids, target_ids)   shape: (context_length,)
    ↓  DataLoader (batching + shuffling)
batch: (batch_size, context_length)
    ↓  GPTModel.forward()
logits: (batch_size, context_length, vocab_size)
    ↓  F.cross_entropy(flatten logits, flatten targets)
scalar loss
    ↓  loss.backward()  →  optimizer.step()
updated weights
```

This one diagram is the entire training loop.

</details>

---
# 🏗️ PART 1: Import the GPT Model

🔗 **Notebook 3 Connection:** We import `GPTModel` from the module you built previously.
If running this notebook standalone, uncomment Option B and paste your GPTModel class.

In [ ]:
# Option A: Import from your models directory (recommended)
import sys
sys.path.append("../models")   # adjust path as needed
from UTILS.model import GPTModel

# Option B: If running standalone, define GPTModel inline here
# (paste your full GPTModel class from Notebook 3)

# Verify it initialises correctly
model_test = GPTModel(GPT_CONFIG_SMALL)
print(model_test)

---
# 🔢 PART 2: Loss Functions — The Training Signal

The model outputs **logits** of shape `(batch, seq_len, vocab_size)`.
We flatten to `(batch × seq_len, vocab_size)` and compute cross-entropy against the flat targets.

This gives the model `vocab_size` training signals at *every sequence position* simultaneously —
far more efficient than predicting one token at a time.

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    """Cross-entropy loss for a single (input, target) batch."""
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)                                            # (B, T, V)
    loss   = F.cross_entropy(logits.flatten(0, 1), target_batch.flatten()) # scalar
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    """Average loss over num_batches batches. Uses all if num_batches=None."""
    if len(data_loader) == 0:
        return float("nan")
    num_batches = min(num_batches or len(data_loader), len(data_loader))
    total = 0.0
    for i, (x, y) in enumerate(data_loader):
        if i >= num_batches:
            break
        total += calc_loss_batch(x, y, model, device).item()
    return total / num_batches

### 🛑 STOP & THINK

Before moving on, answer these:

1. Why do we flatten logits from `(B, T, V)` to `(B×T, V)` before `cross_entropy`?
2. If `vocab_size = 50257` and the initial loss is `10.82`, what does that mean?  
   *(Hint: random guessing over 50257 tokens gives `ln(50257) ≈ 10.82`. This is your baseline.)*
3. What does it mean if training loss drops to 0.1 but validation loss stays at 8.0?

---
# 🧪 PART 3: Sanity Check — Overfit on One Batch

**Best Practice:** Before running any full training loop, verify your setup by overfitting
on a single batch. The loss *must* approach near-zero in ~50 steps.

If it doesn't → the bug is in your model, loss function, or optimizer. Catch it here,
before wasting GPU compute on a full run.

In [ ]:
def trainerV0(model, single_batch, optimizer, device, num_epochs):
    """
    Debug trainer: repeatedly trains on ONE batch.
    Goal: loss approaches ~0, proving model + loss + optimizer are all working.
    This is a sanity check, NOT a real training loop.
    """
    input_batch, target_batch = single_batch
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)

    losses = []
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")
    return losses

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(42)
gpt_test  = GPTModel(GPT_CONFIG_SMALL).to(device)
opt_test  = torch.optim.Adam(gpt_test.parameters(), lr=1e-3)
one_batch = next(iter(DataLoader(train_dataset, batch_size=2, shuffle=False)))

print("Overfitting on a single batch for 50 epochs...")
overfit_losses = trainerV0(gpt_test, one_batch, opt_test, device, num_epochs=50)
print(f"\nFinal loss: {overfit_losses[-1]:.4f}  ← should be close to 0")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(overfit_losses, color="steelblue")
plt.title("Single-Batch Overfit Test — Loss Must Approach Zero")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<details>
<summary>🧾 Whiteboard Cue — why the overfit test works</summary>

**Normal training:** model sees different batches → must generalise → loss stays above zero  
**Overfit test:** model sees the *same* batch every step → must memorise it → loss → 0

If loss doesn't reach ~0 in 50 steps, the bug is one of:
1. **Model architecture** — wrong output shape, broken forward pass
2. **Loss function** — wrong flattening, wrong target alignment
3. **Optimizer** — wrong parameters, learning rate too low

Fix the bug *here* before running Phase 1 or Phase 2.

</details>

---
# 🔤 PART 4: Helper Functions for the Training Loop

Three helpers used inside every trainer:
1. `evaluate_model` — compute train + val loss at evaluation checkpoints
2. `generate_text_simple` — greedy autoregressive decoding to monitor generation quality
3. `generate_and_print_sample` — wrapper that prints a sample after each epoch

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    Greedy autoregressive generation.
    At each step: feed context → get logits → pick argmax → append → repeat.

    idx: (1, T) tensor of token IDs (the prompt)
    """
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]           # crop to context window
        with torch.no_grad():
            logits = model(idx_cond)                # (1, T, vocab_size)
        logits   = logits[:, -1, :]                 # last position: (1, vocab_size)
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (1, 1)
        idx      = torch.cat((idx, idx_next), dim=1)
    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)   # (1, T)


def token_ids_to_text(ids, tokenizer):
    return tokenizer.decode(ids.squeeze(0).tolist())


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    """Average train and val loss over eval_iter batches."""
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    """Generate 50 tokens from start_context and print. Called after each epoch."""
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded      = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model, encoded,
                                         max_new_tokens=50,
                                         context_size=context_size)
    print(token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()

### Quick Test: Pre-Training Gibberish

Before training, the model outputs random noise — this is expected and healthy.

In [ ]:
tok             = tiktoken.encoding_for_model("gpt2")
model_untrained = GPTModel(GPT_CONFIG_SMALL)

print("Pre-training output (expected: gibberish):")
generate_and_print_sample(model_untrained, tok, "cpu", "Be Humble")

---
# 🔄 PART 5: trainerV1 — The Core Training Loop

This is the central function of Phase 1. Every line is deliberately chosen.
Study it carefully — everything in Phase 2 adds to this foundation.

### The Training Loop in Plain English

```
for each epoch:
    for each batch:
        1. zero_grad()              ← clear gradients from last step
        2. loss = forward + CE      ← predict, measure error
        3. loss.backward()          ← compute gradients
        4. optimizer.step()         ← update weights
    
    every eval_freq steps → evaluate on train + val
    every epoch           → generate a text sample
```

In [ ]:
def trainerV1(model, train_loader, val_loader, optimizer, device,
              num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    """
    Clean baseline training loop for autoregressive LM training.

    Returns:
        train_losses, val_losses, tokens_seen  ← for plotting
    """
    train_losses, val_losses, tokens_seen_log = [], [], []
    tokens_seen = 0
    global_step = -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                tokens_seen_log.append(tokens_seen)
                print(f"Ep {epoch+1} | Step {global_step:06d} | "
                      f"Train: {train_loss:.3f} | Val: {val_loss:.3f}")

        print(f"\n── Epoch {epoch+1} sample ──")
        generate_and_print_sample(model, tokenizer, device, start_context)
        print()

    return train_losses, val_losses, tokens_seen_log

In [ ]:
# ── Run Phase 1 training ──────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(123)

model     = GPTModel(GPT_CONFIG_SMALL).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
tok       = tiktoken.encoding_for_model("gpt2")

train_losses, val_losses, tokens_seen_log = trainerV1(
    model, train_loader, val_loader, optimizer, device,
    num_epochs    = 10,
    eval_freq     = 5,
    eval_iter     = 5,
    start_context = "Be Humble",
    tokenizer     = tok,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tokens_seen_log, train_losses, label="Train loss")
ax.plot(tokens_seen_log, val_losses,   label="Val loss", linestyle="--")
ax.set_xlabel("Tokens seen")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training Progress — Phase 1 (trainerV1)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<details>
<summary>🧾 Whiteboard Cue — the training loop as a cycle</summary>

```
┌──────────────────────────────────────────────┐
│  for each batch:                             │
│    optimizer.zero_grad()                     │
│    logits = model(x)                         │
│    loss   = cross_entropy(logits, y)         │
│    loss.backward()    ← fills .grad tensors  │
│    optimizer.step()   ← w = w - lr * grad   │
└──────────────────────────────────────────────┘
         ↓ every eval_freq steps
    evaluate on train + val sets
         ↓ every epoch
    generate text sample
```

**Key question:** Why do we call `zero_grad()` at the START of each step, not the end?  
**Answer:** Because PyTorch *accumulates* gradients by default. We reset before each new batch
to prevent gradients from previous batches contaminating the current update.

</details>

### 🛑 STOP & THINK

1. What would happen if you forgot `optimizer.zero_grad()`?
2. Why do we call `model.eval()` before evaluating and `model.train()` after?
3. The train loss is lower than val loss — is this a problem? When does it become one?
4. If you double the learning rate from `4e-4` to `8e-4`, what do you expect to happen?

---
---
# ⚡ PHASE 2: Scaling Up — GPU Optimization

Phase 1 gave us a working training loop. Phase 2 makes it fast enough to train
models at the scale of GPT-2 and GPT-3.

Optimizations stack in three layers:

```
Section 1: Hardware-level    TF32 → BF16 mixed precision → torch.compile → Flash Attention
Section 2: Algorithm-level   AdamW tuning, cosine LR decay, gradient clipping, accumulation  
Section 3: System-level      Distributed Data Parallel (DDP), multi-GPU, torchrun
```

⚠️ **Phase 2 requires a CUDA GPU.** If you're on CPU, read through for understanding —
the concepts apply directly when you move to Colab or a cloud VM.

In [ ]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True   # suppress non-critical compile warnings

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ! nvidia-smi   # ← uncomment in Colab / cloud environments

---
# 🚀 SECTION 1: GPU Performance Optimization

## 1.1 — TensorCores & TF32 Precision

**What are TensorCores?**  
Modern NVIDIA GPUs (Ampere architecture and later) have specialized units called TensorCores
engineered for matrix multiplication — the core operation in every transformer layer.

**FP32 vs TF32:**

| Format | Bits | Notes |
|--------|------|-------|
| FP32 | 32 | PyTorch default. Full precision. |
| TF32 | 19 (effective) | Crops mantissa of FP32 to match TensorCore format. ~3x matrix multiply speedup. |

One line enables it — zero code changes to the model or training loop:

In [ ]:
# Enable TF32 for all matrix multiplications (uses TensorCores on Ampere+ GPUs)
# Real-world gain: ~3x on matmul ops with negligible accuracy impact.
torch.set_float32_matmul_precision('high')

# Note: actual gain is ~3x not 8x because only matmul is optimised —
# other ops still run in FP32. Still a free win, always enable it.

## 1.2 — Mixed Precision Training (BF16)

Going further: run the **forward pass** in 16-bit precision.

| Format | Exponent bits | Mantissa bits | Verdict |
|--------|--------------|---------------|---------|
| FP32   | 8 | 23 | PyTorch default |
| FP16   | 5 | 10 | Fast, but unstable — small exponent range causes gradient underflow |
| **BF16** | **8** | **7** | **Sweet spot** — same exponent range as FP32, half the memory |

**Rule:** `torch.autocast` wraps the forward pass only. The backward pass stays in FP32.

In [ ]:
# Minimal mixed precision example (snippet — full integration is in trainerV2)
# loss_fn = nn.CrossEntropyLoss()
#
# with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
#     logits = model(x)           # forward pass runs in BF16
#     loss   = loss_fn(...)       # loss computed in BF16
#
# loss.backward()                 # backward pass stays in FP32 — do NOT wrap this
#
# Key insight: BF16 keeps the same exponent range as FP32, so gradients
# don't underflow the way they do in FP16. No gradient scaler needed.

## 1.3 — torch.compile()

A single line for an additional ~1.5–2x speedup:

```python
model = torch.compile(model)
```

**What does it do?**  
PyTorch analyses the entire computation graph and applies **kernel fusion** —
merging multiple GPU operations into single, more efficient kernels.
This cuts memory round-trips between GPU compute units and VRAM.

Example: a GELU activation runs as several separate GPU ops without compile.
With compile, they fuse into one op → one memory read, one write.

**When to use it:** always, unless you are actively debugging.
The first forward pass is slower (compilation overhead), then it accelerates.

## 1.4 — Flash Attention

Standard attention stores the full `(T × T)` attention matrix in GPU memory.
For long sequences this becomes the bottleneck.

**Flash Attention** tiles the attention computation to avoid materialising the full matrix:

```
Standard attention memory: O(T²)   — 1024²= 1M values per head per batch
Flash Attention memory:    O(T)    — never stores the full matrix
Speed:                     2–4x faster for long sequences
Result:                    mathematically identical output
```

PyTorch 2.0+ includes it as `F.scaled_dot_product_attention`:

```python
# Standard (from Notebook 3):
attn_weights = softmax(q @ k.T / scale)
output = attn_weights @ v

# Flash Attention (one line replacement):
output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
```

Let's re-implement the full model with Flash Attention:

In [ ]:
# ── Full GPT model re-implemented with Flash Attention ──────────────────────

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config['emb_dim'], 4 * config['emb_dim']),
            nn.GELU(),
            nn.Linear(4 * config['emb_dim'], config['emb_dim']),
        )
    def forward(self, x):
        return self.layers(x)


class FlashAttention(nn.Module):
    """
    Multi-head attention using F.scaled_dot_product_attention (Flash Attention).
    Key change: replaces explicit (T×T) attention matrix with memory-efficient tiled computation.
    """
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out      = d_out
        self.num_heads  = num_heads
        self.head_dim   = d_out // num_heads
        self.W_query    = nn.Linear(d_in, d_out, bias=bias)
        self.W_key      = nn.Linear(d_in, d_out, bias=bias)
        self.W_value    = nn.Linear(d_in, d_out, bias=bias)
        self.proj       = nn.Linear(d_out, d_out)
        self.dropout_p  = dropout

    def forward(self, x):
        b, t, _ = x.shape
        q = self.W_query(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        # ← Flash Attention: no explicit (T×T) matrix stored
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p = self.dropout_p if self.training else 0.0,
            is_causal = True   # causal mask applied internally
        )
        out = out.transpose(1, 2).contiguous().view(b, t, self.d_out)
        return self.proj(out)


class TransformerBlock_Flash(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention   = FlashAttention(
            d_in=config["emb_dim"], d_out=config["emb_dim"],
            context_length=config["context_length"],
            dropout=config["dropout"], num_heads=config["n_head"],
            bias=config["qkv_bias"],
        )
        self.feedforward = FeedForward(config)
        self.norm1       = nn.LayerNorm(config["emb_dim"])
        self.norm2       = nn.LayerNorm(config["emb_dim"])
        self.dropout     = nn.Dropout(config["dropout"])

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm1(x)))   # pre-norm residual
        x = x + self.dropout(self.feedforward(self.norm2(x)))
        return x


class GPTModel_FLASH(nn.Module):
    """GPT model with Flash Attention, positional embeddings, and final layer norm."""
    def __init__(self, config):
        super().__init__()
        self.token_emb    = nn.Embedding(config['vocab_size'],     config['emb_dim'])
        self.position_emb = nn.Embedding(config['context_length'], config['emb_dim'])
        self.dropout      = nn.Dropout(config['dropout'])
        self.blocks       = nn.Sequential(
            *[TransformerBlock_Flash(config) for _ in range(config['n_layer'])]
        )
        self.norm = nn.LayerNorm(config['emb_dim'])
        self.head = nn.Linear(config['emb_dim'], config['vocab_size'])

    def forward(self, x):
        b, t = x.shape
        tok = self.token_emb(x)
        pos = self.position_emb(torch.arange(t, device=x.device).unsqueeze(0))
        x   = self.dropout(tok + pos)
        x   = self.blocks(x)
        x   = self.norm(x)
        return self.head(x)


print("GPTModel_FLASH defined. Blocks:", len(GPTModel_FLASH(GPT_CONFIG_124M).blocks))

### 💡 Bonus: Power-of-Two Dimensions

GPU hardware is optimised for powers of two. Using "ugly" numbers like 768 for batch size
or 50257 for vocabulary can leave compute units partially idle.

When you control the dimensions (batch size, sequence length, hidden size),
prefer: 16, 32, 64, 128, 256, 512, 1024, 2048 ...

As Andrej Karpathy noted: *"We add calculations, but it goes faster"* — because
the extra aligned compute is faster than misaligned compute on fewer elements.

---
# 🎯 SECTION 2: Algorithmic Optimization & Hyperparameter Tuning

## 2.1 — AdamW with GPT-2/GPT-3 Hyperparameters

The original GPT-2 and GPT-3 training used specific AdamW settings:

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 6e-4,          # peak learning rate
    betas        = (0.9, 0.95),   # β2=0.95, not 0.999
    eps          = 1e-8,
    weight_decay = 0.1,
)
```

**Why β2=0.95?** The default β2=0.999 makes the optimizer very slow to forget old gradients.
For LLM training with millions of steps, β2=0.95 lets the optimizer adapt faster to the
current gradient landscape.

---

## 2.2 — Cosine LR Decay with Linear Warmup

Fixed learning rate → model overshoots at the start, stalls at the end.
**Cosine decay with warmup** is the standard recipe for LLM training:

```
Phase 1 (warmup):  LR rises linearly  0 → max_lr  over first ~1% of steps
Phase 2 (decay):   LR follows cosine  max_lr → min_lr  over remaining steps
```

**Why warmup?** A randomly initialised model has chaotic gradients.
Starting with a tiny LR and ramping up lets the model stabilise before taking large steps.

---

## 2.3 — Gradient Clipping

Occasionally a bad batch produces huge gradients that spike the loss and destabilise training.
Gradient clipping caps the **global gradient norm** before the optimizer step:

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

Called AFTER `loss.backward()`, BEFORE `optimizer.step()`.

---

## 2.4 — Gradient Accumulation: Simulating Large Batch Sizes

GPT-3 was trained with ~3.2 million tokens per batch.
On a single 40GB GPU you might fit batch_size=8.

**Gradient accumulation** bridges this gap:

```
effective_batch = micro_batch × grad_accum_steps × sequence_length

# e.g. micro_batch=16, grad_accum=32, seq_len=1024  →  524,288 tokens/step
```

Critical: divide the loss by `grad_accum_steps` so gradient scale is unchanged.

In [ ]:
import math

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr=None):
    """Cosine decay with linear warmup. Used verbatim by GPT-2 and GPT-3."""
    if min_lr is None:
        min_lr = max_lr * 0.1
    # Phase 1: linear warmup
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    # Phase 2: clamp at min_lr after max_steps
    if step >= max_steps:
        return min_lr
    # Phase 2: cosine decay
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

# Visualise
steps = list(range(1000))
lrs   = [get_lr(s, warmup_steps=100, max_steps=1000, max_lr=6e-4) for s in steps]

plt.figure(figsize=(9, 3))
plt.plot(steps, lrs, color="steelblue")
plt.axvline(100, color="red", linestyle="--", label="End of warmup (step 100)")
plt.title("Cosine LR Decay with Linear Warmup")
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🛑 STOP & THINK

1. Why do we warm up the learning rate instead of starting at max_lr?  
   *(Hint: what do the gradients look like for a randomly initialised model?)*
2. Why do we divide the loss by `grad_accum_steps` during gradient accumulation?  
   What goes wrong if you forget?
3. Gradient clipping clips the *global norm* of all gradients combined, not each
   parameter individually. Why is a global norm clip better than per-parameter clipping?

## trainerV2 — Phase 2 Training Loop

Every addition vs trainerV1 is marked with `[NEW]`:

In [ ]:
def trainerV2(model, train_loader, val_loader, optimizer, device, num_epochs,
              eval_freq, eval_iter, start_context, tokenizer, save_path,
              scheduler=None, grad_accum_steps=1, max_grad_norm=1.0):
    """
    Production training loop — Phase 2.

    New vs trainerV1:
      [NEW] BF16 mixed precision  via torch.autocast + GradScaler
      [NEW] Gradient accumulation  (grad_accum_steps > 1 = larger effective batch)
      [NEW] Gradient clipping      (prevents exploding gradients)
      [NEW] LR scheduler support   (cosine decay)
      [NEW] Per-step timing        (benchmark your optimisations)
      [NEW] Checkpoint saving      (resume from crashes)
    """
    os.makedirs(save_path, exist_ok=True)
    scaler = torch.cuda.amp.GradScaler()  # [NEW] handles FP16/BF16 gradient scaling

    train_losses, val_losses, tokens_seen_log = [], [], []
    tokens_seen = 0
    global_step = -1

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss_accum = 0.0

        for step_in_epoch, (x, y) in enumerate(train_loader):
            step_start = time.time()

            # [NEW] BF16 mixed precision forward pass
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = calc_loss_batch(x, y, model, device)

            # [NEW] Normalise loss for gradient accumulation
            loss = loss / grad_accum_steps
            loss_accum += loss.detach().item()

            # [NEW] Scaled backward pass (handles BF16 gradient scaling)
            scaler.scale(loss).backward()

            # [NEW] Update weights only after accumulating grad_accum_steps gradients
            if (step_in_epoch + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)  # [NEW]
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if scheduler is not None:
                    scheduler.step()                    # [NEW] advance cosine decay

                tokens_seen += x.numel() * grad_accum_steps
                global_step += 1
                step_ms = (time.time() - step_start) * 1000

                if global_step % eval_freq == 0:
                    train_loss, val_loss = evaluate_model(
                        model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    tokens_seen_log.append(tokens_seen)
                    lr_now = optimizer.param_groups[0]['lr']
                    print(f"Ep {epoch+1} | Step {global_step:06d} | "
                          f"Train: {train_loss:.3f} | Val: {val_loss:.3f} | "
                          f"LR: {lr_now:.2e} | {step_ms:.0f}ms/step")
                loss_accum = 0.0

        # [NEW] End-of-epoch: generate sample + save checkpoint
        print(f"\n── Epoch {epoch+1} sample ──")
        generate_and_print_sample(model, tokenizer, device, start_context)
        ckpt = os.path.join(save_path, f"gpt_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), ckpt)
        print(f"Checkpoint saved: {ckpt}\n")

    return train_losses, val_losses, tokens_seen_log

In [ ]:
# ── Run Phase 2 training ──────────────────────────────────────────────────
# Requires a CUDA GPU. On CPU this will be very slow.

torch.manual_seed(123)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Flash Attention model + torch.compile for kernel fusion [NEW]
model_v2 = GPTModel_FLASH(GPT_CONFIG_124M)
model_v2 = torch.compile(model_v2)           # [NEW] compile for kernel fusion
model_v2.to(device)

# AdamW with GPT-2/GPT-3 hyperparameters [NEW]
optimizer_v2 = torch.optim.AdamW(
    model_v2.parameters(),
    lr=6e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1,
)
scheduler_v2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v2, T_max=10)

# Phase 2 uses larger context + power-of-2 batch size
train_ds_v2 = GPT2Dataset(train_path, max_length=512, stride=128)
val_ds_v2   = GPT2Dataset(val_path,   max_length=512, stride=128)
train_dl_v2 = DataLoader(train_ds_v2, batch_size=16, shuffle=True,  drop_last=True)
val_dl_v2   = DataLoader(val_ds_v2,   batch_size=16, shuffle=False, drop_last=True)

start_time = time.time()
train_losses_v2, val_losses_v2, tokens_v2 = trainerV2(
    model_v2, train_dl_v2, val_dl_v2, optimizer_v2, device,
    num_epochs       = 1,
    eval_freq        = 5,
    eval_iter        = 5,
    start_context    = "Be Humble",
    tokenizer        = tok,
    save_path        = "checkpoints",
    scheduler        = scheduler_v2,
    grad_accum_steps = 4,          # simulate 4x effective batch size
    max_grad_norm    = 1.0,
)
elapsed = (time.time() - start_time) / 60
print(f"Training completed in {elapsed:.2f} minutes.")
# Note: first iteration is slower due to torch.compile — it speeds up after.

---
# 🖥️ SECTION 3: Distributed Data Parallel (DDP) & Multi-GPU Training

## When Do You Need DDP?

| Situation | Solution |
|-----------|----------|
| Model fits in one GPU, dataset is large | **Data Parallelism (DDP)** ← this section |
| Model doesn't fit in one GPU | Model Parallelism (not covered here) |
| Multiple machines, each with multiple GPUs | DDP + torchrun across nodes |

## DDP Key Idea

```
┌─────────────────────────────────────────────────────────┐
│  Each GPU gets:  full model copy + unique data subset   │
│                                                         │
│  GPU 0:  forward → backward → grad_0                   │
│  GPU 1:  forward → backward → grad_1                   │
│  GPU 2:  forward → backward → grad_2                   │
│                      ↓                                  │
│         All-Reduce: avg = (grad_0+grad_1+grad_2) / 3   │
│                      ↓                                  │
│  All GPUs update weights with the same avg gradient    │
│  → all model copies stay identical                     │
└─────────────────────────────────────────────────────────┘
```

## ⚠️ DDP Must Run as a Script

DDP launches one process per GPU. This doesn't work inside Jupyter.
The code below is designed to run as a `.py` file via `torchrun`:

```bash
# Single GPU
torchrun --nproc_per_node=1 train_ddp.py

# 4 GPUs on one machine
torchrun --nproc_per_node=4 train_ddp.py

# 2 machines, 4 GPUs each (8 total)
torchrun --nnodes=2 --nproc_per_node=4 --node_rank=0 --master_addr=<ip> train_ddp.py
```

In [ ]:
# ── DDP imports ───────────────────────────────────────────────────────────────
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from torch.distributed import init_process_group, destroy_process_group

# What each import does:
# torch.multiprocessing    → spawn one process per GPU
# DistributedSampler       → split dataset across processes (no overlap)
# DDP                      → wraps model, handles All-Reduce after backward
# init_process_group       → set up NCCL communication backend
# destroy_process_group    → clean up resources when training finishes

### Building Up to DDP Step-by-Step

We start with a simple mock model so you can see exactly what changes:

In [ ]:
# ── BASELINE: single-GPU training (what you already know) ─────────────────────
class MockDataset(Dataset):
    def __init__(self, size=1000):
        self.data    = torch.randn(size, 10)
        self.targets = torch.randint(0, 2, (size,))
    def __len__(self):          return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.targets[idx]

class MockModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 2))
    def forward(self, x): return self.fc(x)

def train_single_gpu(device):
    model     = MockModel().to(device)
    loader    = DataLoader(MockDataset(), batch_size=32, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(3):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x), y)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/3 | Loss: {loss.item():.4f}")

train_single_gpu(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

### Step 1: DDP Setup Function

In [ ]:
def ddp_setup(rank, world_size):
    """
    Initialise the distributed process group. Called once per process.

    rank:       this GPU's index (0, 1, 2, ...)
    world_size: total number of GPUs
    """
    os.environ['MASTER_ADDR'] = 'localhost'   # master node IP
    os.environ['MASTER_PORT'] = '12355'       # communication port
    init_process_group(backend='nccl', rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)               # each process uses its own GPU

### Step 2: DistributedSampler for the DataLoader

In [ ]:
def prepare_ddp_loader(dataset, batch_size):
    """
    DistributedSampler ensures each GPU sees a non-overlapping subset of the data.
    shuffle=False in DataLoader because the sampler handles shuffling.
    """
    sampler = DistributedSampler(dataset, shuffle=True)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler, shuffle=False)

### Step 3: Main Function (one process per GPU)

In [ ]:
def main_ddp(rank, world_size, num_epochs):
    """
    This function runs INDEPENDENTLY on each GPU process.
    `rank` is the GPU index. `world_size` is total GPU count.
    """
    ddp_setup(rank, world_size)

    dataset   = MockDataset()
    loader    = prepare_ddp_loader(dataset, batch_size=32)
    model     = MockModel().to(rank)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # ← THE KEY LINE: wrap model with DDP
    # DDP will All-Reduce gradients across all GPUs after each backward pass
    model = DDP(model, device_ids=[rank])

    for epoch in range(num_epochs):
        loader.sampler.set_epoch(epoch)   # ensures different shuffle each epoch
        model.train()
        for x, y in loader:
            x, y = x.to(rank), y.to(rank)
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x), y)
            loss.backward()               # ← DDP All-Reduces gradients HERE
            optimizer.step()

        if rank == 0:                     # only rank 0 prints (avoid duplicate output)
            print(f"[Rank {rank}] Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}")

    destroy_process_group()

# Launch with mp.spawn (for notebooks/scripts without torchrun)
if __name__ == '__main__':
    world_size = torch.cuda.device_count()
    if world_size >= 2:
        mp.spawn(main_ddp, args=(world_size, 5), nprocs=world_size)
    else:
        print("Single GPU detected. Run on a multi-GPU machine for DDP.")

## trainerV4 — Full DDP Training Loop for GPT

This combines everything: DDP + mixed precision + gradient accumulation +
cosine LR + gradient clipping + crash-recovery checkpointing.

**Designed to run as a script via `torchrun`, not in a notebook.**

New vs trainerV2 (marked `[DDP]`):

In [ ]:
# ── Save this block as train_ddp.py and run with torchrun ─────────────────────

def trainerV4(model, train_loader, val_loader, optimizer, device, num_epochs,
              eval_freq, eval_iter, save_path, scheduler, grad_accum_steps, criterion):
    """
    Full DDP training loop.

    [DDP] model wrapped with DistributedDataParallel
    [DDP] DistributedSampler.set_epoch() for correct per-epoch shuffling
    [DDP] Checkpoint loading to resume from crashes
    [DDP] model.no_sync() on accumulation steps → skip costly All-Reduce mid-accumulation
    [DDP] All logging and saving gated to rank 0 → no duplicate output
    """
    # [DDP] Wrap model — handles gradient synchronisation automatically
    model = DDP(model, device_ids=[device])
    os.makedirs(save_path, exist_ok=True)

    # [DDP] Resume from checkpoint if available
    checkpoint_path = os.path.join(save_path, "latest_checkpoint.pt")
    start_epoch = 0
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch']
        if dist.get_rank() == 0:
            print(f"Resumed from epoch {start_epoch}")

    scaler = torch.cuda.amp.GradScaler()
    train_losses, val_losses = [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loader.sampler.set_epoch(epoch)   # [DDP] re-shuffle per epoch
        optimizer.zero_grad()

        for step_in_epoch, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            is_last_accum = (step_in_epoch + 1) % grad_accum_steps == 0

            # [DDP] no_sync() on accumulation steps: skip All-Reduce until we're ready to update
            ctx = model.no_sync() if not is_last_accum else torch.contextlib.nullcontext()
            with ctx:
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    logits = model(x)
                    loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                loss = loss / grad_accum_steps
                scaler.scale(loss).backward()

            if is_last_accum:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                tokens_seen += x.numel() * grad_accum_steps
                global_step += 1

                # [DDP] only rank 0 prints evaluation results
                if global_step % eval_freq == 0 and dist.get_rank() == 0:
                    train_loss, val_loss = evaluate_model(
                        model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    print(f"[Rank 0] Ep {epoch+1} | Step {global_step:06d} | "
                          f"Train: {train_loss:.3f} | Val: {val_loss:.3f}")

        # [DDP] only rank 0 saves checkpoints — avoid duplicate writes
        if dist.get_rank() == 0:
            print(f"[Rank 0] Epoch {epoch+1} complete.")
            if (epoch + 1) % 5 == 0:
                torch.save({
                    'epoch':                epoch + 1,
                    'model_state_dict':     model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                }, checkpoint_path)
                print(f"[Rank 0] Checkpoint saved at epoch {epoch+1}.")

    return train_losses, val_losses


# ── Entry point for torchrun ───────────────────────────────────────────────────
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_epochs",    type=int,   default=10)
    parser.add_argument("--batch_size",    type=int,   default=16)
    parser.add_argument("--learning_rate", type=float, default=6e-4)
    parser.add_argument("--weight_decay",  type=float, default=0.1)
    parser.add_argument("--eval_freq",     type=int,   default=5)
    parser.add_argument("--eval_iter",     type=int,   default=5)
    parser.add_argument("--grad_accum",    type=int,   default=4)
    parser.add_argument("--save_path",     type=str,   default="checkpoints")
    args = parser.parse_args()

    # torchrun sets LOCAL_RANK automatically — no need to parse it
    init_process_group(backend='nccl')
    local_rank = int(os.environ['LOCAL_RANK'])
    torch.cuda.set_device(local_rank)
    torch.set_float32_matmul_precision('high')   # TF32

    model = GPTModel_FLASH(GPT_CONFIG_124M)
    model = torch.compile(model)                 # kernel fusion
    model.to(local_rank)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=args.learning_rate,
        weight_decay=args.weight_decay, betas=(0.9, 0.95), eps=1e-8,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-5,
    )
    criterion = nn.CrossEntropyLoss()

    train_ds = GPT2Dataset(train_path, max_length=512, stride=128)
    val_ds   = GPT2Dataset(val_path,   max_length=512, stride=128)
    train_dl = prepare_ddp_loader(train_ds, args.batch_size)
    val_dl   = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False)

    trainerV4(
        model, train_dl, val_dl, optimizer, local_rank,
        num_epochs       = args.num_epochs,
        eval_freq        = args.eval_freq,
        eval_iter        = args.eval_iter,
        save_path        = args.save_path,
        scheduler        = scheduler,
        grad_accum_steps = args.grad_accum,
        criterion        = criterion,
    )
    destroy_process_group()

### How to Run trainerV4

```bash
# Single GPU
torchrun --nproc_per_node=1 train_ddp.py --num_epochs 10 --batch_size 16

# 4 GPUs on one machine
torchrun --nproc_per_node=4 train_ddp.py --num_epochs 10 --batch_size 16

# 2 machines, 4 GPUs each (8 GPUs total)
torchrun --nnodes=2 --nproc_per_node=4 --node_rank=0 \
         --master_addr=<master_ip> train_ddp.py
```

**Key DDP vocabulary:**

| Term | Meaning |
|------|---------|
| `LOCAL_RANK` | GPU index on *this machine* (0–3 for a 4-GPU node) |
| `RANK` | Global process index across all machines |
| `WORLD_SIZE` | Total number of processes (GPUs) across all nodes |
| `model.no_sync()` | Skip gradient All-Reduce on accumulation steps → big speedup |
| `sampler.set_epoch(epoch)` | Ensures different data order each epoch on each GPU |

---
# 🎓 Summary — The Complete Training Progression

| Trainer | Key Additions | When to Use |
|---------|--------------|-------------|
| `trainerV0` | Single-batch overfit test | Debugging: verify model + loss work |
| `trainerV1` | Full loop, eval, text generation | CPU / small GPU experiments |
| `trainerV2` | Mixed precision, grad accumulation, clipping, cosine LR, checkpointing | Single GPU production |
| `trainerV4` | DDP, torchrun, distributed sampler, `no_sync`, crash recovery | Multi-GPU production |

---

## ⚡ Optimization Techniques Cheatsheet

| Technique | Code | Gain |
|-----------|------|------|
| TF32 matmul | `torch.set_float32_matmul_precision('high')` | ~3x matrix multiply |
| BF16 mixed precision | `torch.autocast(device_type='cuda', dtype=torch.bfloat16)` | ~2x memory + speed |
| torch.compile | `model = torch.compile(model)` | ~1.5x via kernel fusion |
| Flash Attention | `F.scaled_dot_product_attention(q, k, v, is_causal=True)` | O(T²)→O(T) memory |
| AdamW β₂=0.95 | `betas=(0.9, 0.95)` | Faster gradient adaptation |
| Cosine LR | `get_lr()` or `CosineAnnealingLR` | Smoother convergence |
| Grad clipping | `clip_grad_norm_(params, 1.0)` | Stability on bad batches |
| Grad accumulation | `loss /= N; backward()` × N then step | Simulate large batch |
| DDP + torchrun | `model = DDP(model)` | Linear scaling across GPUs |

---

## 📍 What's Next — Notebook 5

```
You now have: a trained GPT model + checkpoints  ✅

Notebook 5: Fine-Tuning & Inference
    ├── Load a checkpoint
    ├── Fine-tune on a new dataset (instruction tuning, domain adaptation)
    ├── Temperature sampling, top-k, top-p for better generation quality
    └── Evaluate with perplexity and human evaluation
```

---
## 🧠 Practice & Reflection

### Core Concepts
1. Trace one complete training step in `trainerV1` from input tensor to weight update.
   Name every operation in order and the shape at each step.
2. What is the effective batch size when `batch_size=16` and `grad_accum_steps=4` with `seq_len=512`?
3. Explain gradient clipping in your own words. When would you *increase* `max_norm` beyond 1.0?

### Implementation
4. Modify `trainerV1` to plot train/val loss live using `matplotlib` `plt.pause()`.
5. Add an early stopping `patience` parameter to `trainerV1` that halts training if
   val loss hasn't improved for `patience` evaluations.
6. In `trainerV2`, replace `CosineAnnealingLR` with the custom `get_lr()` function above.
   Manually call it each step with `optimizer.param_groups[0]['lr'] = get_lr(step, ...)`.

### Design Thinking
7. You have 4 GPUs and a 10M-token dataset. Design the optimal training setup:
   which `batch_size`, `grad_accum_steps`, and peak `learning_rate` would you choose?
8. A student says: "I'm getting the same val loss on 1 or 4 GPUs, it just runs 4x faster."
   Is this correct? What could go wrong with a naive DDP setup that would make it wrong?

---
# 🎉 Congratulations!

You just completed the hardest notebook in the MyLLM series. 🏆

Here's what you built:

1. **trainerV0** — a debugging tool that proves your stack works before touching a GPU
2. **trainerV1** — a clean, readable training loop you can extend in any direction
3. **trainerV2** — a production single-GPU trainer using the same techniques as GPT-2/GPT-3
4. **trainerV4** — a distributed multi-GPU trainer ready to scale to any number of GPUs

The concepts in this notebook — gradient accumulation, mixed precision, Flash Attention,
cosine LR decay, DDP — are not just tricks. They are the engineering decisions that made
modern large language models possible.

**What's next:** Notebook 5 — Fine-Tuning & Inference 🚀

In [ ]:
def celebrate_end_of_notebook():
    print("🚀 Completing the hardest notebook in the series...")
    for i in range(5, 0, -1):
        print(f"  {i}...")
        time.sleep(1)
    print("\n🎊🎉 We did it! 🎉🎊")
    print("trainerV0 → trainerV1 → trainerV2 → trainerV4 ✅")
    print("Single batch → single GPU → multi-GPU → distributed ✅")
    print("\nWhat's next? Fine-tuning and inference — Notebook 5! 🚀")

celebrate_end_of_notebook()